In [1]:
import cobra

In [ ]:
# Load the model
model = cobra.io.read_sbml_model("model.xml")

In [3]:
# Find the aspartate metaboltie
model.metabolites.cpd00041_c0

Metabolite identifier,cpd00041_c0
Name,L-Aspartate
Memory address,0x117e597d0
Formula,C4H6NO4
Compartment,c0
In 15 reaction(s),"rxn00838_c0, rxn00345_c0, rxn03147_c0, rxn00342_c0, rxn00346_c0, rxn00262_c0, rxn00260_c0, rxn00347_c0, rxn00337_c0, rxn05296_c0, rxn01018_c0, rxn00338_c0, rxn01434_c0, rxn00348_c0, rxn00416_c0"


In [ ]:
# Look at the external glutamate metabolite's reactions to find the transport reaction(s)
model.metabolites.cpd00023_e0.reactions

frozenset({<Reaction EX_cpd00023_e0 at 0x11a0b51d0>,
           <Reaction rxn05298_c0 at 0x11820e950>})

In [ ]:
# Look at the transport reaction for reference
# Does it use Na/H? How much? What genes are associated?
model.reactions.rxn05298_c0

Reaction identifier,rxn05298_c0
Name,Na+/glutamate symport [c0]
Memory address,0x11820e950
Stoichiometry,cpd00023_e0 + cpd00971_e0 <=> cpd00023_c0 + cpd00971_c0 L-Glutamate [e0] + Na+ [e0] <=> L-Glutamate + Na+
GPR,WP_080986407.1
Lower bound,-1000.0
Upper bound,1000.0


In [7]:
# Do we have a Michelle gene ID for WP_080986407.1
model.genes.get_by_id("WP_080986407.1")

Gene identifier,WP_080986407.1
Name,G_WP_080986407.1
Memory address,0x118108c50
Functional,True
In 6 reaction(s),"rxn09247_c0, rxn05495_c0, rxn34491_c0, rxn05298_c0, rxn09306_c0, rxn05684_c0"


In [8]:
# Make an external version of the aspartate metabolite
asp_e0_met = cobra.Metabolite("cpd00041_e0",
    compartment="e0",
    name=model.metabolites.cpd00041_c0.name,
    formula=model.metabolites.cpd00041_c0.formula,
    charge=model.metabolites.cpd00041_c0.charge)
asp_e0_met

Metabolite identifier,cpd00041_e0
Name,L-Aspartate
Memory address,0x119c31310
Formula,C4H6NO4
Compartment,e0
In 0 reaction(s),


In [ ]:
# Add a transport reaction for KDPG
trans = cobra.Reaction("rxn34493")
trans.name = "Aspartate:Na+ symporter"
trans.lower_bound = 0  # Disallow uptake by default
trans.upper_bound = 1000   # Allow secretion
trans.add_metabolites({
    model.metabolites.cpd00041_c0: -1,  # Aspartate in the cytosol
    asp_e0_met: 1,   # Aspartate in the extracellular space
})
# Set the GPR to be the same as the glutamate transport reaction
trans.gene_reaction_rule = model.reactions.rxn05298_c0.gene_reaction_rule
model.add_reactions([trans])

AttributeError: DictList has no attribute or entry trans

In [11]:
model.reactions.rxn34493

Reaction identifier,rxn34493
Name,Aspartate:Na+ symporter
Memory address,0x119c43750
Stoichiometry,cpd00041_c0 --> cpd00041_e0 L-Aspartate --> L-Aspartate
GPR,WP_080986407.1
Lower bound,0
Upper bound,1000


In [12]:
# Add an exchange reaction for the external aspartate
model.add_boundary(model.metabolites.cpd00041_e0, type="exchange")
model.reactions.EX_cpd00041_e0

Reaction identifier,EX_cpd00041_e0
Name,L-Aspartate exchange
Memory address,0x119d24910
Stoichiometry,cpd00041_e0 <=> L-Aspartate <=>
GPR,
Lower bound,-1000.0
Upper bound,1000.0


In [13]:
# Save the model
cobra.io.write_sbml_model(model, "model.xml")
cobra.io.save_json_model(model, "/Users/helenscott/Desktop/model.json")